# Laboratorio con formato Delta Lake

dataset: NYC TLC Trip Record Data (viajes de taxi de Nueva York).

formato Parquet



In [15]:
!pip install -q pyspark==4.0.2 delta-spark==4.0.0

In [16]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

try:
    spark.stop()
except:
    pass

builder = (
    SparkSession.builder
    .appName("EAFIT-PySpark-4.0.2-Delta-4.0.0-Colab")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.0.2


In [17]:
# dataset NYC taxis
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DATASET = "yellow"   # opciones: yellow, green, fhv, hvfhv
YEAR = 2025
MONTHS = [1, 2, 3]

def build_urls(dataset=DATASET, year=YEAR, months=MONTHS):
    return [
        f"{BASE_URL}/{dataset}_tripdata_{year}-{m:02d}.parquet"
        for m in months
    ]

urls = build_urls()
urls


['https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet',
 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet']

In [18]:
import os
import urllib.request
from pathlib import Path

raw_dir = Path("datalake/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

def download_if_missing(url, output_dir=raw_dir):
    filename = url.split("/")[-1]
    target = output_dir / filename
    if not target.exists():
        print(f"Descargando {filename} ...")
        urllib.request.urlretrieve(url, target)
    else:
        print(f"Ya existe: {filename}")
    return str(target)

local_files = [download_if_missing(u) for u in urls]
local_files[:3]


Descargando yellow_tripdata_2025-01.parquet ...
Descargando yellow_tripdata_2025-02.parquet ...
Descargando yellow_tripdata_2025-03.parquet ...


['datalake/raw/yellow_tripdata_2025-01.parquet',
 'datalake/raw/yellow_tripdata_2025-02.parquet',
 'datalake/raw/yellow_tripdata_2025-03.parquet']

# preparación de datos

In [19]:

from pyspark.sql import functions as F
base_df = (
    spark.read.parquet(*local_files)
    .withColumn("year", F.year("tpep_pickup_datetime"))
    .withColumn("month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
)

base_df.show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----+-----+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|year|month|pickup_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----+-----+-----------+
|       1| 2025-03-01 00:17:16|  2025-03-01 00:25:52|        

In [20]:
silver_df = (
    base_df
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
)

silver_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----+-----+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|year|month|pickup_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----+-----+-----------+
|       1| 2025-03-01 00:17:16|  2025-03-01 00:25:52|        

In [21]:
delta_path = "nyc_taxi_yellow"
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .save(delta_path)
)
print(delta_path)


nyc_taxi_yellow


In [22]:

delta_df = spark.read.format("delta").load(delta_path)
delta_df.groupBy("year", "month").count().orderBy("year", "month").show()


+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2007|   12|      1|
|2009|    1|      1|
|2024|   12|     21|
|2025|    1|3253872|
|2025|    2|3310812|
|2025|    3|3848643|
|2025|    4|      2|
+----+-----+-------+



## UPDATE y DELETE

In [23]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)

# Ajuste didáctico: marcar viajes extremos como outlier_flag
delta_table.update(
    condition="trip_distance > 100",
    set={"store_and_fwd_flag": F.lit("OUTLIER")}
)

# Borrar registros obviamente inválidos
delta_table.delete("trip_distance <= 0 OR fare_amount <= 0")


## MERGE con una tabla de correcciones

In [24]:

corrections = spark.createDataFrame(
    [
        (1, 999.99),
        (2, 888.88),
    ],
    ["VendorID", "corrected_total_amount"]
)

# Resumen por Vendor para poder hacer merge controlado
vendor_summary = (
    spark.read.format("delta").load(delta_path)
    .groupBy("VendorID")
    .agg(F.avg("total_amount").alias("avg_total_amount"),
         F.count("*").alias("n_trips"))
)

summary_path = "lakehouse/delta/vendor_summary"
vendor_summary.write.format("delta").mode("overwrite").save(summary_path)

summary_table = DeltaTable.forPath(spark, summary_path)

(
    summary_table.alias("t")
    .merge(corrections.alias("s"), "t.VendorID = s.VendorID")
    .whenMatchedUpdate(set={"avg_total_amount": "s.corrected_total_amount"})
    .whenNotMatchedInsert(values={
        "VendorID": "s.VendorID",
        "avg_total_amount": "s.corrected_total_amount",
        "n_trips": "0"
    })
    .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Historial y time travel

In [25]:

delta_table.history().show(truncate=False)


+-------+-----------------------+------+--------+---------+----------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                                 |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                                                                                                                               

In [26]:

# Ejemplo de lectura por versión
history_df = delta_table.history().select("version").orderBy(F.col("version").desc())
versions = [r["version"] for r in history_df.collect()]
print("Versiones:", versions)

if len(versions) >= 2:
    previous_version = versions[1]
    old_df = spark.read.format("delta").option("versionAsOf", previous_version).load(delta_path)
    print("Conteo versión anterior:", old_df.count())
    print("Conteo versión actual:", spark.read.format("delta").load(delta_path).count())


Versiones: [1, 0]
Conteo versión anterior: 10413352
Conteo versión actual: 10413352
